In [1]:
%%writefile config.py

"""
Production Configuration for HybridMed-AI
Centralized configuration management with environment variable support
"""

import os
from dataclasses import dataclass
import logging

@dataclass
class ModelPaths:
    """Model artifact paths - externalize for container deployment"""
    ML_ENSEMBLE: str = "/kaggle/input/ml-ensemble/transformers/default/1/production_symptom_classifier.pkl"
    PUBMEDBERT: str = "/kaggle/input/fine-tune-pubbert/transformers/default/1/final_PubMedBERT_model"
    LLAMA: str = "/kaggle/input/medical-llama3-1-8b-finetuned-merged-4bit/transformers/default/1/medical_llama3.1_8b_finetuned_merged_4bit"
    RAG_INDEX: str = "/kaggle/input/gale-encyclopidic-knowledge-base"

@dataclass
class ThresholdConfig:
    """Clinical decision thresholds - tuned for safety"""
    ML_CASCADE_THRESHOLD: float = 0.75
    UNCERTAINTY_THRESHOLD: float = 0.70
    EMERGENCY_CONFIDENCE_OVERRIDE: float = 0.60
    
@dataclass
class APIConfig:
    """API and external service configuration"""
    GROQ_API_KEY: str = os.getenv('GROQ_API_KEY', '')
    NGROK_AUTH_TOKEN: str = os.getenv('NGROK_AUTH_TOKEN', '')
    MAX_IMAGE_SIZE_MB: int = 20
    MAX_TOKENS_LLAMA: int = 512
    REQUEST_TIMEOUT_SECONDS: int = 30
    
@dataclass
class SafetyConfig:
    """Medical safety guardrails"""
    ENABLE_EMERGENCY_DETECTION: bool = True
    ENABLE_CONFIDENCE_WARNINGS: bool = True
    REQUIRE_MEDICAL_DISCLAIMER: bool = True
    LOG_ALL_PREDICTIONS: bool = True

# Singleton instances
PATHS = ModelPaths()
THRESHOLDS = ThresholdConfig()
API = APIConfig()
SAFETY = SafetyConfig()

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)


Overwriting config.py


In [2]:
%%writefile groq_service.py

"""
Groq LLM Service - Query Classification, Emergency Detection, Vision Analysis
Production-hardened with comprehensive error handling
"""

from groq import Groq
from typing import Dict, Any, List, Optional
import json
import os
import logging
from PIL import Image
import io
import base64
from enum import Enum

from config import API, SAFETY

logger = logging.getLogger(__name__)

class QueryType(str, Enum):
    """Enumerated query categories for type-safe routing"""
    SYMPTOM = "symptom"
    GENERAL_MEDICAL = "general_medical"
    EMERGENCY = "emergency"
    IRRELEVANT = "irrelevant"
    SYSTEM_ERROR = "system_error"

class EmergencyCategory(str, Enum):
    """Clinical emergency categories"""
    CARDIAC = "cardiac"
    RESPIRATORY = "respiratory"
    NEUROLOGICAL = "neurological"
    HEMORRHAGE = "hemorrhage"
    TRAUMA = "trauma"
    PSYCHIATRIC = "psychiatric"

class GroqService:
    """
    Production LLM service for query processing and classification.
    
    This service handles:
    - Medical query classification (symptom vs general vs emergency)
    - Emergency red flag detection using clinical criteria
    - Medical image analysis with vision models
    - Fallback mechanisms for API failures
    """
    
    def __init__(self):
        """Initialize Groq client with error handling"""
        api_key = API.GROQ_API_KEY
        
        if not api_key:
            logger.error("GROQ_API_KEY not configured")
            raise ValueError("Missing GROQ_API_KEY - cannot initialize Groq service")
            
        try:
            self.client = Groq(api_key=api_key)
            self.text_model = 'llama-3.1-8b-instant'
            self.vision_model = 'meta-llama/llama-4-scout-17b-16e-instruct'
            logger.info(f"✅ Groq service initialized")
        except Exception as e:
            logger.error(f"Failed to initialize Groq client: {e}")
            raise
        
        self._initialize_emergency_patterns()
    
    def _initialize_emergency_patterns(self):
        """Initialize clinical emergency detection patterns"""
        self.emergency_patterns = {
            EmergencyCategory.CARDIAC: {
                "patterns": [
                    "chest pain", "crushing chest", "pressure in chest",
                    "tightness in chest", "heart attack", "cardiac arrest"
                ],
                "severity": "CRITICAL",
                "action": "Call 911 immediately - Possible heart attack"
            },
            EmergencyCategory.RESPIRATORY: {
                "patterns": [
                    "can't breathe", "difficulty breathing", "gasping for air",
                    "struggling to breathe", "choking", "severe shortness of breath"
                ],
                "severity": "CRITICAL",
                "action": "Call 911 immediately - Respiratory emergency"
            },
            EmergencyCategory.NEUROLOGICAL: {
                "patterns": [
                    "face drooping", "arm weakness", "slurred speech",
                    "sudden confusion", "worst headache of life",
                    "thunderclap headache", "seizure", "unresponsive"
                ],
                "severity": "CRITICAL",
                "action": "Call 911 immediately - Possible stroke"
            },
            EmergencyCategory.HEMORRHAGE: {
                "patterns": [
                    "vomiting blood", "coughing up blood", "blood in stool",
                    "severe bleeding", "uncontrolled bleeding"
                ],
                "severity": "CRITICAL",
                "action": "Call 911 immediately - Severe bleeding"
            },
            EmergencyCategory.PSYCHIATRIC: {
                "patterns": [
                    "want to die", "suicidal", "kill myself",
                    "end my life", "harm myself"
                ],
                "severity": "CRITICAL",
                "action": "Call 911 or National Suicide Prevention Lifeline: 988"
            }
        }
    
    def detect_emergency(self, query: str) -> Dict[str, Any]:
        """Detect medical emergencies using pattern matching"""
        if not SAFETY.ENABLE_EMERGENCY_DETECTION:
            return {"is_emergency": False}
        
        query_lower = query.lower()
        detected_emergencies = []
        
        for category, config in self.emergency_patterns.items():
            matched_patterns = []
            
            for pattern in config["patterns"]:
                if pattern in query_lower:
                    matched_patterns.append(pattern)
            
            if matched_patterns:
                detected_emergencies.append({
                    "category": category.value,
                    "matched_patterns": matched_patterns,
                    "severity": config["severity"],
                    "action": config["action"],
                    "confidence": min(1.0, len(matched_patterns) * 0.3)
                })
        
        if detected_emergencies:
            primary_emergency = max(detected_emergencies, key=lambda x: x["confidence"])
            
            logger.warning(f"🚨 EMERGENCY DETECTED: {primary_emergency['category']}")
            
            return {
                "is_emergency": True,
                "primary_emergency": primary_emergency,
                "all_detected": detected_emergencies,
                "emergency_message": self._format_emergency_message(primary_emergency)
            }
        
        return {"is_emergency": False}
    
    def _format_emergency_message(self, emergency: Dict) -> str:
        """Format standardized emergency alert message"""
        return f"""🚨 **MEDICAL EMERGENCY DETECTED**

**Category:** {emergency['category'].upper()}
**Severity:** {emergency['severity']}

**IMMEDIATE ACTION REQUIRED:**
{emergency['action']}

**DO NOT WAIT:**
- Do not attempt self-treatment
- Do not drive yourself if symptoms are severe
- Stay with someone if possible

⚠️ This AI detected emergency indicators. Professional emergency medical evaluation is critical."""
    
    def classify_query(self, query: str) -> Dict[str, Any]:
        """Classify medical query type using LLM"""
        
        # Check for emergencies first
        emergency_check = self.detect_emergency(query)
        
        if emergency_check["is_emergency"]:
            return {
                "query_type": QueryType.EMERGENCY,
                "confidence": emergency_check["primary_emergency"]["confidence"],
                "reasoning": "Emergency patterns detected",
                "symptoms_detected": [],
                "clean_query": query,
                "emergency_info": emergency_check,
                "route_to": "emergency_handler"
            }
        
        # LLM classification for non-emergency queries
        system_prompt = """You are a medical query classifier.

Classify into ONE category:

1. "symptom" - User describes personal symptoms
   Examples: "I have headache and fever", "My chest hurts"
   
2. "general_medical" - User asks medical questions
   Examples: "What causes diabetes?", "How to prevent flu?"
   
3. "irrelevant" - Not medical
   Examples: "What's the weather?", "Tell me a joke"

Return ONLY valid JSON:
{
  "category": "symptom" | "general_medical" | "irrelevant",
  "confidence": 0.0-1.0,
  "reasoning": "brief explanation",
  "symptoms_detected": ["list"],
  "clean_query": "normalized query"
}"""
        
        try:
            response = self.client.chat.completions.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Classify: {query}"}
                ],
                model=self.text_model,
                temperature=0.2,
                max_tokens=300,
                response_format={"type": "json_object"}
            )
            
            result = json.loads(response.choices[0].message.content)
            query_type = QueryType(result.get("category", "general_medical"))
            
            route_mapping = {
                QueryType.SYMPTOM: "ml_dl_pipeline",
                QueryType.GENERAL_MEDICAL: "llm_rag",
                QueryType.IRRELEVANT: "rejection",
                QueryType.EMERGENCY: "emergency_handler"
            }
            
            return {
                "query_type": query_type,
                "confidence": result.get("confidence", 0.8),
                "reasoning": result.get("reasoning", ""),
                "symptoms_detected": result.get("symptoms_detected", []),
                "clean_query": result.get("clean_query", query),
                "emergency_info": None,
                "route_to": route_mapping[query_type]
            }
            
        except Exception as e:
            logger.error(f"Query classification failed: {e}")
            return self._fallback_classification(query)
    
    def _fallback_classification(self, query: str) -> Dict[str, Any]:
        """Fallback classification when LLM fails"""
        query_lower = query.lower()
        
        symptom_keywords = ["i have", "i feel", "my", "pain", "ache", "hurt"]
        general_keywords = ["what is", "what are", "how does", "what causes"]
        
        if any(kw in query_lower for kw in symptom_keywords):
            category = QueryType.SYMPTOM
        elif any(kw in query_lower for kw in general_keywords):
            category = QueryType.GENERAL_MEDICAL
        else:
            category = QueryType.GENERAL_MEDICAL
        
        logger.warning(f"Using fallback classification: {category}")
        
        return {
            "query_type": category,
            "confidence": 0.6,
            "reasoning": "Fallback keyword-based classification",
            "symptoms_detected": [],
            "clean_query": query,
            "emergency_info": None,
            "route_to": "ml_dl_pipeline" if category == QueryType.SYMPTOM else "llm_rag"
        }
    
    def analyze_medical_image(
        self, 
        image_bytes: bytes, 
        user_query: str = "",
        filename: str = ""
    ) -> Dict[str, Any]:
        """Analyze medical images using vision model"""
        try:
            image = Image.open(io.BytesIO(image_bytes))
            
            if image.format not in ['JPEG', 'PNG', 'JPG']:
                return {
                    "success": False,
                    "error": "UNSUPPORTED_FORMAT",
                    "message": f"Image format '{image.format}' not supported. Use JPEG or PNG."
                }
            
            image_size_mb = len(image_bytes) / (1024 * 1024)
            if image_size_mb > API.MAX_IMAGE_SIZE_MB:
                return {
                    "success": False,
                    "error": "IMAGE_TOO_LARGE",
                    "message": f"Image size ({image_size_mb:.1f}MB) exceeds limit"
                }
            
            image = image.convert("RGB")
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=85)
            image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
            
            prompt = f"""You are a medical AI assistant analyzing a clinical image.

User's question: {user_query if user_query else 'Please analyze this medical image'}

Provide a structured analysis:

1. **Visual Observations**
   - Describe what you see

2. **Differential Diagnosis**
   - List possible conditions

3. **Severity Assessment**
   - Mild / Moderate / Severe / Uncertain

4. **Clinical Recommendations**
   - When to seek medical attention

IMPORTANT: Acknowledge limitations and recommend professional evaluation."""
            
            response = self.client.chat.completions.create(
                model=self.vision_model,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}
                        }
                    ]
                }],
                temperature=0.3,
                max_tokens=800
            )
            
            analysis = response.choices[0].message.content
            
            full_analysis = f"""📸 **Medical Image Analysis**

{analysis}

---
⚠️ **Disclaimer**: This is AI-based visual assessment and should NOT replace professional medical diagnosis."""
            
            logger.info(f"✅ Vision analysis complete")
            
            return {
                "success": True,
                "analysis": full_analysis,
                "filename": filename,
                "image_size_mb": image_size_mb
            }
            
        except Exception as e:
            logger.error(f"Vision analysis failed: {e}")
            
            return {
                "success": False,
                "error": "VISION_ANALYSIS_FAILED",
                "message": f"Could not analyze image: {str(e)}",
                "filename": filename
            }


Overwriting groq_service.py


In [3]:
%%writefile ml_model.py

"""
ML Ensemble Model - Production Implementation
Handles symptom-to-disease classification with stacked ensemble architecture
"""

import joblib
import numpy as np
from typing import List, Tuple, Dict, Optional
import os
import logging
from dataclasses import dataclass
from datetime import datetime
import threading

logger = logging.getLogger(__name__)


@dataclass
class PredictionResult:
    """Structured prediction result with metadata"""
    disease: str
    confidence: float
    rank: int
    timestamp: datetime
    model_version: str


@dataclass
class ModelMetadata:
    """Model metadata for reproducibility and auditing"""
    model_type: str
    n_diseases: int
    training_date: Optional[str]
    performance_metrics: Dict
    feature_dims: Dict


class MLEnsembleModel:
    """
    Production ML Ensemble for symptom-to-disease classification.
    
    Architecture:
    - Stacking ensemble of base learners (SVM, XGBoost, DT, RF, NB)
    - Dual feature extraction (word-level + character-level n-grams)
    - Meta-learner for final prediction
    
    Thread Safety: Uses lock for model access
    Deterministic: Fixed random seeds, reproducible predictions
    Validated: Comprehensive input validation
    """
    
    def __init__(self, model_path: str):
        """Initialize ML model with path validation"""
        if not model_path:
            raise ValueError("model_path cannot be empty")
        
        self.model_path = model_path
        self.model = None
        self.label_encoder = None
        self.word_vectorizer = None
        self.char_vectorizer = None
        self.disease_labels = []
        self.metadata = ModelMetadata(
            model_type="Unknown",
            n_diseases=0,
            training_date=None,
            performance_metrics={},
            feature_dims={}
        )
        self._lock = threading.Lock()
        self._is_loaded = False
    
    def load(self) -> bool:
        """
        Load trained ensemble model with comprehensive validation.
        
        Returns:
            True if successful, raises exception otherwise
        """
        with self._lock:
            logger.info(f"Loading ML ensemble from: {self.model_path}")
            
            # Validation: Check file exists
            if not os.path.exists(self.model_path):
                raise FileNotFoundError(
                    f"Model artifact not found: {self.model_path}\n"
                    f"Ensure the model file is available at the specified path."
                )
            
            # Load artifact
            try:
                artifacts = joblib.load(self.model_path)
            except Exception as e:
                raise RuntimeError(f"Failed to deserialize model file: {e}")
            
            # Validation: Required keys
            required_keys = ["model", "label_encoder", "feature_vectorizers"]
            missing_keys = [k for k in required_keys if k not in artifacts]
            if missing_keys:
                raise KeyError(
                    f"Model artifact missing required keys: {missing_keys}\n"
                    f"Available keys: {list(artifacts.keys())}"
                )
            
            # Extract components
            self.model = artifacts["model"]
            self.label_encoder = artifacts["label_encoder"]
            
            # Extract vectorizers
            vectorizers = artifacts["feature_vectorizers"]
            if not isinstance(vectorizers, dict):
                raise ValueError("feature_vectorizers must be a dictionary")
            
            self.word_vectorizer = vectorizers.get("word")
            self.char_vectorizer = vectorizers.get("char")
            
            if self.word_vectorizer is None or self.char_vectorizer is None:
                raise ValueError(
                    "Both 'word' and 'char' vectorizers must be present\n"
                    f"Found: {list(vectorizers.keys())}"
                )
            
            # Validation: Check vectorizers are fitted
            self._validate_vectorizers()
            
            # Extract disease labels
            self.disease_labels = list(self.label_encoder.classes_)
            
            # Extract metadata
            raw_metadata = artifacts.get("metadata", {})
            performance = artifacts.get("performance_metrics", {})
            
            self.metadata = ModelMetadata(
                model_type=type(self.model).__name__,
                n_diseases=len(self.disease_labels),
                training_date=raw_metadata.get("training_date"),
                performance_metrics=performance,
                feature_dims={
                    "word_vocab": len(self.word_vectorizer.vocabulary_),
                    "char_vocab": len(self.char_vectorizer.vocabulary_)
                }
            )
            
            # Validation: Check predict_proba exists
            if not hasattr(self.model, "predict_proba"):
                raise TypeError(
                    f"Model type {type(self.model).__name__} does not support "
                    f"predict_proba(). Confidence scores unavailable."
                )
            
            self._is_loaded = True
            
            # Log success
            logger.info("✅ ML Ensemble loaded successfully")
            logger.info(f"   • Model Type: {self.metadata.model_type}")
            logger.info(f"   • Diseases: {self.metadata.n_diseases}")
            logger.info(f"   • Word Vocab: {self.metadata.feature_dims['word_vocab']}")
            logger.info(f"   • Char Vocab: {self.metadata.feature_dims['char_vocab']}")
            
            if self.metadata.performance_metrics:
                test_f1 = self.metadata.performance_metrics.get('test_f1', 0)
                test_acc = self.metadata.performance_metrics.get('test_accuracy', 0)
                logger.info(f"   • Test F1: {test_f1:.3f}")
                logger.info(f"   • Test Accuracy: {test_acc:.2%}")
            
            return True
    
    def _validate_vectorizers(self):
        """Validate that vectorizers are properly fitted"""
        try:
            if not hasattr(self.word_vectorizer, 'vocabulary_'):
                raise ValueError("Word vectorizer is not fitted")
            
            if len(self.word_vectorizer.vocabulary_) == 0:
                raise ValueError("Word vectorizer has empty vocabulary")
            
            if not hasattr(self.char_vectorizer, 'vocabulary_'):
                raise ValueError("Char vectorizer is not fitted")
            
            if len(self.char_vectorizer.vocabulary_) == 0:
                raise ValueError("Char vectorizer has empty vocabulary")
            
            logger.debug("✓ Vectorizers validated successfully")
            
        except Exception as e:
            logger.error(f"Vectorizer validation failed: {e}")
            raise
    
    def predict(
        self, 
        symptoms: List[str], 
        top_k: int = 3
    ) -> List[Tuple[str, float]]:
        """
        Predict diseases from symptom list.
        
        This method expects RAW SYMPTOM TEXT. The StackingClassifier contains
        internal pipelines that handle feature extraction automatically.
        
        Args:
            symptoms: List of symptom strings (e.g., ["headache", "fever"])
            top_k: Number of top predictions to return
            
        Returns:
            List of (disease, confidence) tuples, sorted by confidence
        """
        if not self._is_loaded:
            raise RuntimeError(
                "Model not loaded. Call load() before making predictions."
            )
        
        # Input validation
        if not symptoms:
            raise ValueError("symptoms list cannot be empty")
        
        if not isinstance(symptoms, list):
            raise TypeError(f"symptoms must be a list, got {type(symptoms)}")
        
        # Sanitize inputs
        symptoms = [str(s).strip() for s in symptoms if s]
        if not symptoms:
            raise ValueError("No valid symptoms after sanitization")
        
        try:
            with self._lock:
                # Prepare text input
                text = " ".join(symptoms).lower().strip()
                
                if not text:
                    text = "none"
                
                # Model expects list of strings (StackingClassifier with pipelines)
                probs = self.model.predict_proba([text])[0]
                
                # Get top-k indices
                top_k = min(top_k, len(self.disease_labels))
                top_indices = np.argsort(probs)[-top_k:][::-1]
                
                # Build results
                predictions = [
                    (self.disease_labels[i], float(probs[i]))
                    for i in top_indices
                ]
                
                # Audit log
                logger.info(f"Prediction: {predictions[0][0]} ({predictions[0][1]:.2%})")
                
                return predictions
                
        except Exception as e:
            logger.error(f"Prediction failed: {e}", exc_info=True)
            
            # Graceful degradation: Return safe fallback
            logger.warning("Returning fallback predictions due to error")
            return self._get_fallback_predictions()
    
    def _get_fallback_predictions(self) -> List[Tuple[str, float]]:
        """Fallback predictions when model fails"""
        return [
            ("Common Cold", 0.45),
            ("Viral Infection", 0.30),
            ("Flu", 0.25)
        ]
    
    def predict_with_confidence_check(
        self,
        symptoms: List[str],
        cascade_threshold: float = 0.75
    ) -> Dict:
        """
        Predict with explicit confidence gating for cascading.
        
        Args:
            symptoms: List of symptom strings
            cascade_threshold: Confidence threshold (cascade if below)
            
        Returns:
            Dictionary with prediction details and cascade decision
        """
        predictions = self.predict(symptoms, top_k=3)
        top_disease, top_confidence = predictions[0]
        
        should_cascade = top_confidence < cascade_threshold
        
        cascade_reason = None
        if should_cascade:
            cascade_reason = (
                f"ML confidence ({top_confidence:.2%}) below threshold "
                f"({cascade_threshold:.2%}). Deep learning cascade recommended."
            )
        
        return {
            "disease": top_disease,
            "confidence": top_confidence,
            "all_predictions": predictions,
            "should_cascade": should_cascade,
            "cascade_reason": cascade_reason,
            "model_type": "ML_ENSEMBLE"
        }
    
    def get_model_info(self) -> Dict:
        """Get comprehensive model information for debugging and auditing"""
        return {
            "model_type": self.metadata.model_type,
            "n_diseases": self.metadata.n_diseases,
            "disease_labels": self.disease_labels,
            "training_date": self.metadata.training_date,
            "performance_metrics": self.metadata.performance_metrics,
            "feature_dimensions": self.metadata.feature_dims,
            "is_loaded": self._is_loaded,
            "model_path": self.model_path
        }
    
    def validate_model(self) -> Dict[str, bool]:
        """Validate model integrity for health checks"""
        checks = {
            "model_loaded": self._is_loaded,
            "model_exists": self.model is not None,
            "label_encoder_exists": self.label_encoder is not None,
            "word_vectorizer_exists": self.word_vectorizer is not None,
            "char_vectorizer_exists": self.char_vectorizer is not None,
            "has_disease_labels": len(self.disease_labels) > 0,
            "supports_predict_proba": hasattr(self.model, "predict_proba") if self.model else False
        }
        
        all_passed = all(checks.values())
        checks["all_checks_passed"] = all_passed
        
        return checks


Overwriting ml_model.py


In [4]:
%%writefile pubbert_model.py

"""
PubMedBERT Deep Learning Model - Production Implementation
Fine-tuned biomedical language model for symptom classification
"""

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import List, Tuple, Dict, Optional
import logging
from dataclasses import dataclass
import threading
import gc

logger = logging.getLogger(__name__)


@dataclass
class ModelConfig:
    """PubMedBERT configuration"""
    max_length: int = 512
    batch_size: int = 8
    temperature: float = 1.0
    

class PubMedBERTModel:
    """
    Production PubMedBERT model for deep learning cascade.
    
    Features:
    - GPU memory management with automatic cleanup
    - Batch inference support for efficiency
    - Thread-safe prediction
    - Graceful degradation on OOM errors
    - Model warmup for consistent latency
    """
    
    def __init__(self, model_path: str, disease_labels: List[str] = None):
        """Initialize PubMedBERT model"""
        if not model_path:
            raise ValueError("model_path cannot be empty")
        
        self.model_path = model_path
        self.disease_labels = disease_labels or []
        self.model = None
        self.tokenizer = None
        self.device = None
        self.config = ModelConfig()
        self._lock = threading.Lock()
        self._is_loaded = False
        self._warmup_done = False
    
    def load(self) -> bool:
        """Load fine-tuned PubMedBERT model with validation"""
        with self._lock:
            try:
                logger.info(f"Loading PubMedBERT from: {self.model_path}")
                
                # Determine device
                self.device = "cuda" if torch.cuda.is_available() else "cpu"
                logger.info(f"Using device: {self.device}")
                
                if self.device == "cuda":
                    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
                    logger.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
                
                # Load tokenizer
                logger.info("Loading tokenizer...")
                self.tokenizer = AutoTokenizer.from_pretrained(
                    self.model_path,
                    use_fast=True
                )
                
                # Load model
                logger.info("Loading model...")
                self.model = AutoModelForSequenceClassification.from_pretrained(
                    self.model_path,
                    num_labels=len(self.disease_labels) if self.disease_labels else 24,
                    ignore_mismatched_sizes=True,
                    torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
                )
                
                # Move to device
                self.model.to(self.device)
                
                # Set to evaluation mode
                self.model.eval()
                
                # Disable gradient computation (inference only)
                for param in self.model.parameters():
                    param.requires_grad = False
                
                self._is_loaded = True
                
                if not self.disease_labels:
                    logger.warning("No disease labels provided - predictions will use indices")
                
                logger.info("✅ PubMedBERT loaded successfully")
                logger.info(f"   • Device: {self.device}")
                logger.info(f"   • Disease Labels: {len(self.disease_labels)}")
                logger.info(f"   • Max Length: {self.config.max_length}")
                
                # Warmup
                self._warmup()
                
                return True
                
            except torch.cuda.OutOfMemoryError:
                logger.error("GPU Out of Memory during model loading")
                self._cleanup_gpu()
                raise RuntimeError(
                    "Insufficient GPU memory to load PubMedBERT. "
                    "Try reducing batch size or using CPU."
                )
            
            except Exception as e:
                logger.error(f"Failed to load PubMedBERT: {e}", exc_info=True)
                raise
    
    # def _warmup(self):
    #     """Warmup model with dummy input to stabilize latency"""
    #     if self._warmup_done:
    #         return
        
    #     try:
    #         logger.info("Warming up model...")
    #         dummy_text = "patient presents with headache and fever"
    #         _ = self.predict([dummy_text], top_k=1)
    #         self._warmup_done = True
    #         logger.info("✓ Model warmup complete")
    #     except Exception as e:
    #         logger.warning(f"Warmup failed (non-critical): {e}")
    def _warmup(self):
        """Skip warmup - optional performance optimization"""
        self._warmup_done = True
        logger.info("✓ Warmup skipped (optional - first inference may be slower)")


    
    def predict(
        self,
        symptoms: List[str],
        top_k: int = 3
    ) -> List[Tuple[str, float]]:
        """
        Predict diseases from symptoms using PubMedBERT.
        
        Args:
            symptoms: List of symptom strings
            top_k: Number of top predictions
            
        Returns:
            List of (disease, confidence) tuples
        """
        if not self._is_loaded:
            raise RuntimeError("Model not loaded. Call load() first.")
        
        if not self.disease_labels:
            logger.error("No disease labels available for prediction")
            return []
        
        # Input validation
        if not symptoms:
            raise ValueError("symptoms list cannot be empty")
        
        symptoms = [str(s).strip() for s in symptoms if s]
        if not symptoms:
            raise ValueError("No valid symptoms after sanitization")
        
        try:
            with self._lock:
                # Prepare input text with medical formatting
                text = "Patient presents with: " + ", ".join(symptoms)
                
                # Tokenize
                inputs = self.tokenizer(
                    text,
                    return_tensors="pt",
                    max_length=self.config.max_length,
                    truncation=True,
                    padding=True
                )
                
                # Move to device
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                
                # Inference with no gradient
                with torch.no_grad():
                    outputs = self.model(**inputs)
                    logits = outputs.logits
                    
                    # Apply temperature scaling
                    logits = logits / self.config.temperature
                    
                    # Softmax to get probabilities
                    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
                
                # Get top-k predictions
                top_k = min(top_k, len(self.disease_labels))
                top_indices = np.argsort(probs)[-top_k:][::-1]
                
                predictions = [
                    (self.disease_labels[i], float(probs[i]))
                    for i in top_indices
                ]
                
                # Audit log
                logger.info(f"DL Prediction: {predictions[0][0]} ({predictions[0][1]:.2%})")
                
                return predictions
                
        except torch.cuda.OutOfMemoryError:
            logger.error("GPU OOM during inference")
            self._cleanup_gpu()
            logger.warning("Returning empty predictions due to GPU OOM")
            return []
        
        except Exception as e:
            logger.error(f"PubMedBERT prediction failed: {e}", exc_info=True)
            return []
    
    def _cleanup_gpu(self):
        """Clean up GPU memory"""
        if self.device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
            logger.info("GPU memory cleaned up")
    
    def get_model_info(self) -> Dict:
        """Get model information"""
        return {
            "model_type": "PubMedBERT",
            "model_path": self.model_path,
            "device": str(self.device),
            "n_diseases": len(self.disease_labels),
            "disease_labels": self.disease_labels,
            "max_length": self.config.max_length,
            "is_loaded": self._is_loaded,
            "warmup_done": self._warmup_done
        }
    
    def validate_model(self) -> Dict[str, bool]:
        """Validate model for health checks"""
        checks = {
            "model_loaded": self._is_loaded,
            "model_exists": self.model is not None,
            "tokenizer_exists": self.tokenizer is not None,
            "has_disease_labels": len(self.disease_labels) > 0,
            "device_available": self.device is not None,
            "warmup_complete": self._warmup_done
        }
        
        checks["all_checks_passed"] = all(checks.values())
        return checks


Overwriting pubbert_model.py


In [5]:
%%writefile llama_service.py

"""
LLaMA 3.1 Service - Production Implementation
Fine-tuned medical reasoning model with 4-bit quantization
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from typing import List, Optional, Dict
import logging
import threading
import time

logger = logging.getLogger(__name__)


class GenerationConfig:
    """LLaMA generation configuration"""
    MAX_NEW_TOKENS = 400
    TEMPERATURE = 0.7
    TOP_P = 0.9
    TOP_K = 50
    REPETITION_PENALTY = 1.2
    NO_REPEAT_NGRAM_SIZE = 3
    GENERATION_TIMEOUT = 30


class LLaMAService:
    """
    Production LLaMA 3.1 service for medical explanation generation.
    
    Features:
    - 4-bit quantization for memory efficiency
    - Anti-hallucination controls (repetition penalty, n-gram blocking)
    - Timeout handling for generation
    - Grounded in retrieved context (RAG integration)
    - Medical disclaimer injection
    """
    
    def __init__(self, model_path: str):
        """Initialize LLaMA service"""
        if not model_path:
            raise ValueError("model_path cannot be empty")
        
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self.device = None
        self.config = GenerationConfig()
        self._lock = threading.Lock()
        self._is_loaded = False
    
    def load(self) -> bool:
        """Load fine-tuned LLaMA model with 4-bit quantization"""
        with self._lock:
            try:
                logger.info(f"Loading fine-tuned LLaMA 3.1 from: {self.model_path}")
                logger.info("This may take 2-3 minutes with quantization...")
                
                # Device
                self.device = "cuda" if torch.cuda.is_available() else "cpu"
                logger.info(f"Using device: {self.device}")
                
                # 4-bit quantization config (for GPU efficiency)
                bnb_config = None
                if self.device == "cuda":
                    bnb_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16,
                        bnb_4bit_use_double_quant=True
                    )
                    logger.info("Using 4-bit quantization (NF4)")
                
                # Load tokenizer
                logger.info("Loading tokenizer...")
                self.tokenizer = AutoTokenizer.from_pretrained(
                    self.model_path,
                    use_fast=True
                )
                
                # Ensure pad token exists
                if self.tokenizer.pad_token is None:
                    self.tokenizer.pad_token = self.tokenizer.eos_token
                
                # Load model
                logger.info("Loading model (this is the slow part)...")
                load_kwargs = {
                    "trust_remote_code": True,
                    "low_cpu_mem_usage": True
                }
                
                if bnb_config:
                    load_kwargs["quantization_config"] = bnb_config
                    load_kwargs["device_map"] = "auto"
                else:
                    load_kwargs["torch_dtype"] = torch.float32
                
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    **load_kwargs
                )
                
                if not bnb_config:
                    self.model.to(self.device)
                
                self.model.eval()
                
                self._is_loaded = True
                
                logger.info("✅ Fine-tuned LLaMA loaded successfully")
                logger.info(f"   • Device: {self.device}")
                logger.info(f"   • Quantization: {'4-bit NF4' if bnb_config else 'None (FP32)'}")
                
                return True
                
            except Exception as e:
                logger.error(f"Failed to load LLaMA: {e}", exc_info=True)
                raise
    
    def generate_explanation(
        self,
        disease: str,
        context: str = "",
        symptoms: Optional[List[str]] = None
    ) -> str:
        """
        Generate medical explanation grounded in context.
        
        Args:
            disease: Predicted disease name
            context: Retrieved context from RAG
            symptoms: Original symptom list
            
        Returns:
            Generated explanation with medical disclaimer
        """
        if not self._is_loaded:
            logger.warning("LLaMA not loaded - returning fallback")
            return self._get_fallback_explanation(disease)
        
        try:
            with self._lock:
                # Build prompt with context grounding
                symptoms_text = ", ".join(symptoms) if symptoms else "various symptoms"
                
                prompt = f"""Based on the symptoms: {symptoms_text}

The condition appears to be: {disease}

Medical Context:
{context if context else "No additional context available."}

Provide a clear, evidence-based explanation covering:
1. What is {disease}?
2. Common symptoms and causes
3. When to seek medical attention

Explanation:"""
                
                # Tokenize
                inputs = self.tokenizer(
                    prompt,
                    return_tensors="pt",
                    max_length=1024,
                    truncation=True
                )
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                
                # Generate with timeout
                start_time = time.time()
                
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=self.config.MAX_NEW_TOKENS,
                        temperature=self.config.TEMPERATURE,
                        top_p=self.config.TOP_P,
                        top_k=self.config.TOP_K,
                        do_sample=True,
                        repetition_penalty=self.config.REPETITION_PENALTY,
                        no_repeat_ngram_size=self.config.NO_REPEAT_NGRAM_SIZE,
                        pad_token_id=self.tokenizer.eos_token_id,
                        eos_token_id=self.tokenizer.eos_token_id
                    )
                
                generation_time = time.time() - start_time
                
                # Check timeout
                if generation_time > self.config.GENERATION_TIMEOUT:
                    logger.warning(f"Generation took {generation_time:.1f}s (timeout: {self.config.GENERATION_TIMEOUT}s)")
                
                # Decode
                generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # Extract explanation (after "Explanation:")
                if "Explanation:" in generated_text:
                    explanation = generated_text.split("Explanation:")[-1].strip()
                else:
                    explanation = generated_text
                
                # Limit length to prevent excessive repetition
                sentences = explanation.split('. ')
                if len(sentences) > 8:
                    explanation = '. '.join(sentences[:8]) + '.'
                
                # Add medical disclaimer
                explanation += "\n\n⚠️ **Medical Disclaimer**: This AI-generated information is for educational purposes only and should not replace professional medical advice. Always consult a healthcare provider for diagnosis and treatment."
                
                logger.info(f"Generated explanation ({len(explanation)} chars in {generation_time:.1f}s)")
                
                return explanation
                
        except Exception as e:
            logger.error(f"Generation failed: {e}", exc_info=True)
            return self._get_fallback_explanation(disease)
    
    def _get_fallback_explanation(self, disease: str) -> str:
        """Fallback explanation when generation fails"""
        return f"""Based on the provided symptoms, you may have **{disease}**.

**Important**: This is an automated assessment. Please consult a healthcare professional for:
- Accurate diagnosis through clinical examination
- Appropriate treatment recommendations
- Monitoring and follow-up care

**When to seek immediate care**:
- Symptoms worsen rapidly
- High fever persists
- Severe pain or discomfort
- Difficulty breathing

⚠️ **Disclaimer**: AI predictions have limitations and cannot replace professional medical evaluation."""
    
    def get_model_info(self) -> Dict:
        """Get model information"""
        return {
            "model_type": "LLaMA-3.1-8B-Fine-tuned",
            "model_path": self.model_path,
            "device": str(self.device),
            "quantization": "4-bit NF4" if self.device == "cuda" else "None",
            "max_new_tokens": self.config.MAX_NEW_TOKENS,
            "temperature": self.config.TEMPERATURE,
            "is_loaded": self._is_loaded
        }
    
    def validate_model(self) -> Dict[str, bool]:
        """Validate model for health checks"""
        checks = {
            "model_loaded": self._is_loaded,
            "model_exists": self.model is not None,
            "tokenizer_exists": self.tokenizer is not None,
            "device_available": self.device is not None
        }
        
        checks["all_checks_passed"] = all(checks.values())
        return checks


Overwriting llama_service.py


In [6]:
%%writefile rag_service.py

"""
RAG (Retrieval-Augmented Generation) Service - Production Implementation
FAISS-based semantic search over medical encyclopedia
"""

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import json
import os
import logging
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import threading

logger = logging.getLogger(__name__)


@dataclass
class RetrievalResult:
    """Structured retrieval result"""
    text: str
    score: float
    source: str
    rank: int


class RAGService:
    """
    Production RAG system for medical knowledge retrieval.
    
    Features:
    - FAISS vector search for efficient retrieval
    - Relevance scoring and filtering
    - Citation-style provenance tracking
    - Fallback mechanisms for low-confidence retrieval
    """
    
    def __init__(self, index_path: str):
        """Initialize RAG service"""
        if not index_path:
            raise ValueError("index_path cannot be empty")
        
        self.index_path = index_path
        self.index = None
        self.documents = []
        self.embedder = None
        self._lock = threading.Lock()
        self._is_loaded = False
        
        # Configuration
        self.relevance_threshold = 0.3
        self.min_documents = 1
    
    def load(self) -> bool:
        """Load FAISS index and documents"""
        with self._lock:
            try:
                logger.info(f"Loading RAG system from: {self.index_path}")
                
                # Load FAISS index
                index_file = os.path.join(self.index_path, "index.faiss")
                if not os.path.exists(index_file):
                    raise FileNotFoundError(f"FAISS index not found: {index_file}")
                
                logger.info("Loading FAISS index...")
                self.index = faiss.read_index(index_file)
                
                # Load documents
                docs_file = os.path.join(self.index_path, "documents.json")
                if not os.path.exists(docs_file):
                    raise FileNotFoundError(f"Documents not found: {docs_file}")
                
                logger.info("Loading documents...")
                with open(docs_file, 'r', encoding='utf-8') as f:
                    self.documents = json.load(f)
                
                # Validate document structure
                if not self.documents:
                    raise ValueError("Documents list is empty")
                
                if not isinstance(self.documents, list):
                    raise TypeError("Documents must be a list")
                
                # Validate index-document alignment
                if self.index.ntotal != len(self.documents):
                    logger.warning(
                        f"Index size ({self.index.ntotal}) != document count ({len(self.documents)}). "
                        f"This may cause retrieval errors."
                    )
                
                # Load embedder
                logger.info("Loading sentence embedder...")
                self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
                
                self._is_loaded = True
                
                logger.info("✅ RAG system loaded successfully")
                logger.info(f"   • Documents: {len(self.documents):,}")
                logger.info(f"   • Index Size: {self.index.ntotal:,}")
                logger.info(f"   • Dimension: {self.index.d}")
                
                return True
                
            except Exception as e:
                logger.error(f"Failed to load RAG: {e}", exc_info=True)
                raise
    
    def retrieve(
        self,
        query: str,
        top_k: int = 3,
        return_scores: bool = False
    ) -> str:
        """
        Retrieve relevant context for query.
        
        Args:
            query: Search query
            top_k: Number of documents to retrieve
            return_scores: Include relevance scores in output
            
        Returns:
            Concatenated context string
        """
        if not self._is_loaded:
            logger.warning("RAG not loaded - returning empty context")
            return ""
        
        if not query or not query.strip():
            logger.warning("Empty query provided to RAG")
            return ""
        
        try:
            with self._lock:
                # Encode query
                query_embedding = self.embedder.encode([query], convert_to_numpy=True)
                
                # Search FAISS index
                distances, indices = self.index.search(
                    query_embedding.astype('float32'),
                    top_k
                )
                
                # Extract documents
                contexts = []
                retrieval_results = []
                
                for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
                    # Validate index
                    if idx < 0 or idx >= len(self.documents):
                        logger.warning(f"Invalid index {idx} (max: {len(self.documents)-1})")
                        continue
                    
                    doc = self.documents[idx]
                    
                    # Calculate similarity score (FAISS returns L2 distance)
                    similarity = 1.0 / (1.0 + distance)
                    
                    # Filter by relevance threshold
                    if similarity < self.relevance_threshold:
                        logger.debug(f"Filtered document {idx}: similarity {similarity:.3f} < {self.relevance_threshold}")
                        continue
                    
                    # Extract text
                    text = doc.get('text', '')
                    source = doc.get('source', 'Unknown')
                    
                    if not text:
                        logger.warning(f"Document {idx} has no text field")
                        continue
                    
                    retrieval_results.append(
                        RetrievalResult(
                            text=text,
                            score=similarity,
                            source=source,
                            rank=rank
                        )
                    )
                    
                    # Format context
                    if return_scores:
                        contexts.append(f"[Relevance: {similarity:.2f}] {text}")
                    else:
                        contexts.append(text)
                
                # Check if minimum documents retrieved
                if len(contexts) < self.min_documents:
                    logger.warning(
                        f"Only {len(contexts)} relevant documents found (min: {self.min_documents}). "
                        f"Query may be too specific or out of domain."
                    )
                
                # Join contexts
                combined_context = "\n\n".join(contexts)
                
                logger.info(f"Retrieved {len(contexts)} documents for query: '{query[:50]}...'")
                
                return combined_context
                
        except Exception as e:
            logger.error(f"RAG retrieval failed: {e}", exc_info=True)
            return ""
    
    def retrieve_with_metadata(
        self,
        query: str,
        top_k: int = 3
    ) -> List[RetrievalResult]:
        """Retrieve with full metadata for citation purposes"""
        if not self._is_loaded:
            return []
        
        try:
            with self._lock:
                query_embedding = self.embedder.encode([query], convert_to_numpy=True)
                distances, indices = self.index.search(
                    query_embedding.astype('float32'),
                    top_k
                )
                
                results = []
                
                for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
                    if idx < 0 or idx >= len(self.documents):
                        continue
                    
                    doc = self.documents[idx]
                    similarity = 1.0 / (1.0 + distance)
                    
                    if similarity < self.relevance_threshold:
                        continue
                    
                    results.append(
                        RetrievalResult(
                            text=doc.get('text', ''),
                            score=similarity,
                            source=doc.get('source', 'Unknown'),
                            rank=rank
                        )
                    )
                
                return results
                
        except Exception as e:
            logger.error(f"Retrieval with metadata failed: {e}", exc_info=True)
            return []
    
    def get_system_info(self) -> Dict:
        """Get RAG system information"""
        return {
            "index_path": self.index_path,
            "n_documents": len(self.documents),
            "index_size": self.index.ntotal if self.index else 0,
            "dimension": self.index.d if self.index else 0,
            "embedder": "sentence-transformers/all-MiniLM-L6-v2",
            "relevance_threshold": self.relevance_threshold,
            "is_loaded": self._is_loaded
        }
    
    def validate_system(self) -> Dict[str, bool]:
        """Validate RAG system for health checks"""
        checks = {
            "system_loaded": self._is_loaded,
            "index_exists": self.index is not None,
            "documents_loaded": len(self.documents) > 0,
            "embedder_exists": self.embedder is not None,
            "index_documents_aligned": self.index.ntotal == len(self.documents) if self.index else False
        }
        
        checks["all_checks_passed"] = all(checks.values())
        return checks


Overwriting rag_service.py


In [7]:
%%writefile schemas.py

"""
Pydantic Schemas for Type-Safe API
Production-grade request/response models
"""

from pydantic import BaseModel, Field, ConfigDict, field_validator
from typing import List, Optional, Dict, Any
from enum import Enum
from datetime import datetime


class QueryType(str, Enum):
    """Query classification types"""
    SYMPTOM = "symptom"
    GENERAL_MEDICAL = "general_medical"
    EMERGENCY = "emergency"
    IRRELEVANT = "irrelevant"


class ModelType(str, Enum):
    """Model types for tracking which system handled request"""
    ML_ENSEMBLE = "ml_ensemble"
    PUBMEDBERT = "pubmedbert"
    LOCAL_LLAMA = "local_llama"
    GROQ_LLAMA = "groq_llama"


class SeverityLevel(str, Enum):
    """Medical severity levels"""
    UNKNOWN = "unknown"
    MILD = "mild"
    MODERATE = "moderate"
    SEVERE = "severe"
    CRITICAL = "critical"


# Request Models

class ChatRequest(BaseModel):
    """Main chat request"""
    query: str = Field(..., min_length=1, max_length=2000, description="User's medical query")
    use_cascade: bool = Field(default=True, description="Enable ML->DL cascade")
    patient_age: Optional[int] = Field(default=None, ge=0, le=150, description="Patient age (optional)")
    patient_gender: Optional[str] = Field(default=None, description="Patient gender (optional)")
    
    @field_validator('query')
    @classmethod
    def validate_query(cls, v):
        """Validate query is not just whitespace"""
        if not v.strip():
            raise ValueError("Query cannot be empty or whitespace only")
        return v.strip()


class ImageAnalysisRequest(BaseModel):
    """Image analysis request"""
    message: str = Field(default="", max_length=500, description="Context about the image")
    filename: str = Field(default="", max_length=255, description="Original filename")


# Response Models

class PredictionResult(BaseModel):
    """Individual prediction result"""
    disease: str = Field(..., description="Predicted disease name")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Prediction confidence")
    probability: float = Field(..., ge=0.0, le=1.0, description="Model probability")
    rank: int = Field(..., ge=1, description="Rank in prediction list")


class EmergencyInfo(BaseModel):
    """Emergency detection information"""
    is_emergency: bool
    category: Optional[str] = None
    matched_patterns: List[str] = Field(default_factory=list)
    severity: Optional[str] = None
    recommended_action: Optional[str] = None
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Main chat response"""
    model_config = ConfigDict(protected_namespaces=())
    
    success: bool = Field(..., description="Request success status")
    query_processed: str = Field(..., description="Cleaned/processed query")
    query_type: QueryType = Field(..., description="Classified query type")
    model_used: ModelType = Field(..., description="Primary model used")
    
    predictions: List[PredictionResult] = Field(default_factory=list, description="Disease predictions")
    explanation: Optional[str] = Field(default=None, description="Generated explanation")
    
    confidence_flag: Optional[str] = Field(default=None, description="Confidence warning if applicable")
    emergency_info: Optional[EmergencyInfo] = Field(default=None, description="Emergency details if detected")
    
    cascade_triggered: bool = Field(default=False, description="Whether DL cascade was triggered")
    cascade_reason: Optional[str] = Field(default=None, description="Reason for cascading")
    
    inference_time_ms: float = Field(..., description="Total inference time in milliseconds")
    timestamp: datetime = Field(default_factory=datetime.utcnow, description="Response timestamp")
    
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata")


class ImageAnalysisResponse(BaseModel):
    """Image analysis response"""
    success: bool
    filename: str
    analysis: Optional[str] = None
    error_message: Optional[str] = None
    image_size_mb: Optional[float] = None
    inference_time_ms: float
    timestamp: datetime = Field(default_factory=datetime.utcnow)


class HealthCheckResponse(BaseModel):
    """Health check response"""
    status: str = Field(..., description="Overall system status")
    models: Dict[str, bool] = Field(..., description="Individual model health")
    configuration: Dict[str, Any] = Field(..., description="System configuration")
    uptime_seconds: Optional[float] = Field(default=None, description="System uptime")
    timestamp: datetime = Field(default_factory=datetime.utcnow)


class ErrorResponse(BaseModel):
    """Error response"""
    error: bool = Field(default=True)
    error_type: str = Field(..., description="Error category")
    message: str = Field(..., description="Error message")
    details: Optional[Dict[str, Any]] = Field(default=None, description="Additional error details")
    timestamp: datetime = Field(default_factory=datetime.utcnow)


Overwriting schemas.py


In [8]:
%%writefile main.py

"""
HybridMed-AI FastAPI Application - Production Implementation
Main application with intelligent routing and comprehensive error handling
"""

from fastapi import FastAPI, HTTPException, File, UploadFile, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError

from schemas import (
    ChatRequest, ChatResponse, ImageAnalysisResponse,
    HealthCheckResponse, ErrorResponse,
    ModelType, QueryType, PredictionResult, EmergencyInfo
)

from typing import List, Optional, Dict
import time
import logging
import asyncio
from datetime import datetime
import io
import PyPDF2

# Import services
from config import PATHS, THRESHOLDS, SAFETY
from groq_service import GroqService, QueryType as GroqQueryType
from ml_model import MLEnsembleModel
from pubbert_model import PubMedBERTModel
from llama_service import LLaMAService
from rag_service import RAGService

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


# =============================================================================
# APPLICATION INITIALIZATION
# =============================================================================

app = FastAPI(
    title="HybridMed-AI",
    description="Production-Grade Hybrid Medical AI System with Intelligent Query Routing",
    version="3.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# CORS Configuration
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Restrict in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global service instances
groq: Optional[GroqService] = None
ml_model: Optional[MLEnsembleModel] = None
pubbert: Optional[PubMedBERTModel] = None
llama: Optional[LLaMAService] = None
rag: Optional[RAGService] = None

# System state
system_start_time = None


# =============================================================================
# EXCEPTION HANDLERS
# =============================================================================

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request, exc):
    """Handle Pydantic validation errors"""
    logger.error(f"Validation error: {exc}")
    return JSONResponse(
        status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
        content={
            "error": True,
            "error_type": "VALIDATION_ERROR",
            "message": "Invalid request format",
            "details": exc.errors()
        }
    )


@app.exception_handler(Exception)
async def general_exception_handler(request, exc):
    """Handle uncaught exceptions"""
    logger.error(f"Unhandled exception: {exc}", exc_info=True)
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={
            "error": True,
            "error_type": "INTERNAL_ERROR",
            "message": "An internal error occurred. Please try again later.",
            "details": None
        }
    )


# =============================================================================
# STARTUP & SHUTDOWN
# =============================================================================

@app.on_event("startup")
async def startup_event():
    """Initialize all models on startup"""
    global groq, ml_model, pubbert, llama, rag, system_start_time
    
    system_start_time = time.time()
    
    logger.info("=" * 70)
    logger.info("🚀 INITIALIZING HYBRIDMED-AI v3.0")
    logger.info("=" * 70)
    
    try:
        # 1. Groq Service
        logger.info("\n1️⃣ Initializing Groq Service...")
        groq = GroqService()
        logger.info("✅ Groq Service initialized")
        
        # 2. ML Ensemble
        logger.info("\n2️⃣ Loading ML Ensemble...")
        ml_model = MLEnsembleModel(PATHS.ML_ENSEMBLE)
        ml_model.load()
        
        # 3. PubMedBERT
        logger.info("\n3️⃣ Loading PubMedBERT...")
        pubbert = PubMedBERTModel(
            model_path=PATHS.PUBMEDBERT,
            disease_labels=ml_model.disease_labels
        )
        pubbert.load()
        
        # 4. LLaMA
        logger.info("\n4️⃣ Loading Fine-Tuned LLaMA 3.1...")
        llama = LLaMAService(PATHS.LLAMA)
        llama.load()
        
        # 5. RAG
        logger.info("\n5️⃣ Loading RAG System...")
        rag = RAGService(PATHS.RAG_INDEX)
        rag.load()
        
        logger.info("\n" + "=" * 70)
        logger.info("✅ ALL SYSTEMS OPERATIONAL")
        logger.info("=" * 70)
        
    except Exception as e:
        logger.critical(f"❌ STARTUP FAILED: {e}", exc_info=True)
        raise


@app.on_event("shutdown")
async def shutdown_event():
    """Cleanup on shutdown"""
    logger.info("Shutting down HybridMed-AI...")
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            logger.info("GPU memory cleared")
    except:
        pass
    
    logger.info("Shutdown complete")


# =============================================================================
# HEALTH CHECK ENDPOINTS
# =============================================================================

@app.get("/", response_model=Dict)
async def root():
    """Root endpoint with system information"""
    return {
        "project": "HybridMed-AI",
        "version": "3.0.0",
        "status": "operational",
        "description": "Production-grade hybrid medical AI system",
        "models": [
            "ML Ensemble (Stacking)",
            "PubMedBERT (Fine-tuned)",
            "LLaMA 3.1 (Fine-tuned)",
            "RAG (FAISS)",
            "LLaMA 3.2 Vision (Groq)"
        ],
        "endpoints": {
            "chat": "/chat",
            "image_analysis": "/chat-with-attachments",
            "health": "/health",
            "model_info": "/model-info"
        }
    }


@app.get("/health", response_model=HealthCheckResponse)
async def health_check():
    """Comprehensive health check"""
    uptime = time.time() - system_start_time if system_start_time else 0
    
    models_health = {
        "groq": groq is not None,
        "ml_ensemble": ml_model is not None and ml_model._is_loaded,
        "pubmedbert": pubbert is not None and pubbert._is_loaded,
        "llama": llama is not None and llama._is_loaded,
        "rag": rag is not None and rag._is_loaded
    }
    
    overall_status = "healthy" if all(models_health.values()) else "degraded"
    
    return HealthCheckResponse(
        status=overall_status,
        models=models_health,
        configuration={
            "ml_cascade_threshold": THRESHOLDS.ML_CASCADE_THRESHOLD,
            "uncertainty_threshold": THRESHOLDS.UNCERTAINTY_THRESHOLD,
            "emergency_detection_enabled": SAFETY.ENABLE_EMERGENCY_DETECTION,
            "rag_documents": len(rag.documents) if rag else 0
        },
        uptime_seconds=uptime
    )


@app.get("/model-info")
async def model_info():
    """Detailed model information"""
    info = {}
    
    if ml_model:
        info["ml_ensemble"] = ml_model.get_model_info()
    
    if pubbert:
        info["pubmedbert"] = pubbert.get_model_info()
    
    if llama:
        info["llama"] = llama.get_model_info()
    
    if rag:
        info["rag"] = rag.get_system_info()
    
    return info
# =============================================================================
# MAIN CHAT ENDPOINT
# =============================================================================

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Main chat endpoint with intelligent routing.
    
    Routing Logic:
    - Emergency queries → Immediate alert + medical context
    - Symptom queries → ML/DL pipeline (with cascading)
    - General medical queries → LLM/RAG
    - Irrelevant queries → Rejection with guidance
    """
    start_time = time.time()
    
    try:
        logger.info("=" * 70)
        logger.info(f"📥 Query: {request.query}")
        logger.info("=" * 70)
        
        # =====================================================================
        # STEP 1: QUERY CLASSIFICATION
        # =====================================================================
        
        classification = groq.classify_query(request.query)
        
        query_type_str = classification["query_type"]
        if isinstance(query_type_str, GroqQueryType):
            query_type_str = query_type_str.value
        
        query_type = QueryType(query_type_str)
        clean_query = classification["clean_query"]
        confidence = classification["confidence"]
        symptoms = classification.get("symptoms_detected", [])
        
        logger.info(f"🔍 Classification: {query_type.value}")
        logger.info(f"📍 Route: {classification['route_to']}")
        logger.info(f"✓ Confidence: {confidence:.2f}")
        
        # =====================================================================
        # STEP 2: HANDLE EMERGENCY
        # =====================================================================
        
        if query_type == QueryType.EMERGENCY:
            logger.warning("🚨 EMERGENCY DETECTED")
            
            emergency_data = classification.get("emergency_info", {})
            
            emergency_msg = emergency_data.get("emergency_message", "")
            emergency_msg += "\n\n**Emergency Response Protocol:**\n"
            emergency_msg += "1. Call emergency services (911 or local equivalent) immediately\n"
            emergency_msg += "2. Do not wait for symptoms to improve\n"
            emergency_msg += "3. Stay calm and follow dispatcher instructions\n"
            emergency_msg += "4. If possible, have someone stay with you\n"
            
            # Still provide medical context if symptoms available
            if symptoms:
                try:
                    ml_preds = ml_model.predict(symptoms, top_k=1)
                    if ml_preds:
                        context_disease = ml_preds[0][0]
                        emergency_msg += f"\n**Possible Condition:** {context_disease}\n"
                        
                        context = rag.retrieve(context_disease, top_k=1)
                        if context:
                            emergency_msg += f"\n**Medical Context:**\n{context}\n"
                except:
                    pass
            
            return ChatResponse(
                success=True,
                query_processed=clean_query,
                query_type=query_type,
                model_used=ModelType.ML_ENSEMBLE,
                predictions=[],
                explanation=emergency_msg,
                emergency_info=EmergencyInfo(
                    is_emergency=True,
                    category=emergency_data.get("primary_emergency", {}).get("category"),
                    matched_patterns=emergency_data.get("primary_emergency", {}).get("matched_patterns", []),
                    severity="CRITICAL",
                    recommended_action=emergency_data.get("primary_emergency", {}).get("action"),
                    confidence=emergency_data.get("primary_emergency", {}).get("confidence", 1.0)
                ),
                inference_time_ms=(time.time() - start_time) * 1000
            )
        
        # =====================================================================
        # STEP 3: HANDLE IRRELEVANT QUERIES
        # =====================================================================
        
        if query_type == QueryType.IRRELEVANT:
            logger.info("❌ Irrelevant query detected")
            
            guidance = """I'm HybridMed-AI, a medical AI assistant designed to help with health-related questions.

**I can help with:**
• Symptom analysis and disease prediction
• Medical information and health education
• Treatment options and prevention strategies
• General health questions

**Examples of questions I can answer:**
• "I have a headache and fever, what could it be?"
• "What are the symptoms of diabetes?"
• "How can I prevent heart disease?"

Please ask a medical or health-related question, and I'll be happy to assist you."""
            
            return ChatResponse(
                success=False,
                query_processed=clean_query,
                query_type=query_type,
                model_used=ModelType.GROQ_LLAMA,
                predictions=[],
                explanation=guidance,
                inference_time_ms=(time.time() - start_time) * 1000
            )
        
        # =====================================================================
        # STEP 4: SYMPTOM-BASED QUERIES → ML/DL PIPELINE
        # =====================================================================
        
        if query_type == QueryType.SYMPTOM:
            logger.info("🩺 Routing to ML/DL Pipeline")
            
            # ML Prediction
            logger.info("   → Running ML Ensemble...")
            ml_result = ml_model.predict_with_confidence_check(
                symptoms,
                cascade_threshold=THRESHOLDS.ML_CASCADE_THRESHOLD
            )
            
            predictions_list = ml_result["all_predictions"]
            ml_confidence = ml_result["confidence"]
            should_cascade = ml_result["should_cascade"]
            
            logger.info(f"   ✓ ML Result: {ml_result['disease']} ({ml_confidence:.2%})")
            
            model_used = ModelType.ML_ENSEMBLE
            cascade_triggered = False
            cascade_reason = None
            
            # DL Cascade if needed
            if should_cascade and request.use_cascade:
                logger.info(f"   ⚠️ ML Confidence ({ml_confidence:.2%}) < Threshold")
                logger.info("   → Cascading to PubMedBERT...")
                
                try:
                    dl_preds = pubbert.predict(symptoms, top_k=3)
                    
                    if dl_preds and len(dl_preds) > 0:
                        dl_confidence = dl_preds[0][1]
                        logger.info(f"   ✓ DL Result: {dl_preds[0][0]} ({dl_confidence:.2%})")
                        
                        if dl_confidence > ml_confidence:
                            logger.info(f"   ✅ Using DL predictions")
                            predictions_list = dl_preds
                            model_used = ModelType.PUBMEDBERT
                            cascade_triggered = True
                            cascade_reason = f"DL confidence ({dl_confidence:.2%}) > ML confidence ({ml_confidence:.2%})"
                        else:
                            logger.info(f"   → Keeping ML predictions")
                            cascade_triggered = True
                            cascade_reason = "DL cascade attempted but ML remained more confident"
                    else:
                        logger.warning("   ⚠️ DL returned empty predictions")
                        cascade_triggered = True
                        cascade_reason = "DL cascade attempted but failed"
                
                except Exception as e:
                    logger.error(f"   ❌ DL cascade error: {e}")
                    cascade_triggered = True
                    cascade_reason = f"DL cascade failed: {str(e)}"
            
            # Convert to PredictionResult objects
            predictions = [
                PredictionResult(
                    disease=disease,
                    confidence=conf,
                    probability=conf,
                    rank=i+1
                )
                for i, (disease, conf) in enumerate(predictions_list)
            ]
            
            # Check uncertainty
            final_confidence = predictions[0].confidence
            is_uncertain = final_confidence < THRESHOLDS.UNCERTAINTY_THRESHOLD
            
            confidence_flag = None
            if is_uncertain:
                confidence_flag = f"UNCERTAIN (Confidence: {final_confidence:.1%} < {THRESHOLDS.UNCERTAINTY_THRESHOLD:.1%})"
                logger.warning(f"   ⚠️ {confidence_flag}")
            
            # Generate explanation
            logger.info(f"   → Retrieving context for: {predictions[0].disease}")
            context = rag.retrieve(predictions[0].disease, top_k=3)
            
            logger.info("   → Generating explanation with LLaMA...")
            explanation = llama.generate_explanation(
                disease=predictions[0].disease,
                context=context,
                symptoms=symptoms
            )
            
            # Add uncertainty warning
            if is_uncertain:
                uncertainty_warning = f"""\n\n⚠️ **UNCERTAIN PREDICTION** (Confidence: {final_confidence:.1%})

The model's confidence is below {THRESHOLDS.UNCERTAINTY_THRESHOLD:.0%}, indicating uncertainty in the diagnosis. This could mean:
• Symptoms match multiple conditions
• Insufficient symptom information provided
• Rare or complex condition

**Strong Recommendation:** Please consult a healthcare professional for proper evaluation and diagnosis.
"""
                explanation = uncertainty_warning + explanation
            
            logger.info(f"✅ Prediction Complete: {predictions[0].disease}")
        
        # =====================================================================
        # STEP 5: GENERAL MEDICAL QUERIES → LLM/RAG
        # =====================================================================
        
        else:  # QueryType.GENERAL_MEDICAL
            logger.info("📚 Routing to LLM/RAG Pipeline")
            
            logger.info(f"   → Retrieving context for: {clean_query}")
            context = rag.retrieve(clean_query, top_k=5)
            
            logger.info("   → Generating answer with LLaMA...")
            explanation = llama.generate_explanation(
                disease=clean_query,
                context=context,
                symptoms=[]
            )
            
            predictions = []
            model_used = ModelType.LOCAL_LLAMA
            cascade_triggered = False
            cascade_reason = None
            confidence_flag = None
        
        # =====================================================================
        # STEP 6: BUILD RESPONSE
        # =====================================================================
        
        inference_time = (time.time() - start_time) * 1000
        logger.info(f"⏱️ Total Time: {inference_time:.0f}ms")
        logger.info("=" * 70)
        
        return ChatResponse(
            success=True,
            query_processed=clean_query,
            query_type=query_type,
            model_used=model_used,
            predictions=predictions,
            explanation=explanation,
            confidence_flag=confidence_flag,
            cascade_triggered=cascade_triggered,
            cascade_reason=cascade_reason,
            inference_time_ms=inference_time,
            metadata={
                "patient_age": request.patient_age,
                "patient_gender": request.patient_gender,
                "symptoms_detected": symptoms
            }
        )
    
    except Exception as e:
        logger.error(f"❌ Chat endpoint error: {e}", exc_info=True)
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=str(e)
        )


# =============================================================================
# IMAGE ANALYSIS ENDPOINT
# =============================================================================

@app.post("/chat-with-attachments", response_model=ImageAnalysisResponse)
async def chat_with_attachments(
    files: List[UploadFile] = File(...),
    message: str = ""
):
    """
    Handle medical image analysis with attachments.
    
    Supports: JPEG, PNG images
    """
    start_time = time.time()
    
    try:
        logger.info(f"📎 Processing {len(files)} attachment(s)")
        
        results = []
        
        for file in files:
            file_type = file.content_type
            file_content = await file.read()
            
            logger.info(f"   → File: {file.filename} ({file_type}, {len(file_content)/1024:.1f} KB)")
            
            # Only process images
            if file_type and file_type.startswith("image/"):
                result = groq.analyze_medical_image(
                    image_bytes=file_content,
                    user_query=message,
                    filename=file.filename
                )
                
                if result["success"]:
                    results.append(result["analysis"])
                    logger.info(f"   ✓ Image analysis complete")
                else:
                    error_msg = result.get("message", "Unknown error")
                    results.append(f"[{file.filename}] Could not analyze: {error_msg}")
                    logger.warning(f"   ✗ Image analysis failed: {error_msg}")
            else:
                results.append(f"[{file.filename}] Unsupported file type: {file_type}")
        
        combined_analysis = "\n\n---\n\n".join(results)
        
        inference_time = (time.time() - start_time) * 1000
        
        return ImageAnalysisResponse(
            success=True,
            filename=files[0].filename if files else "",
            analysis=combined_analysis,
            inference_time_ms=inference_time
        )
    
    except Exception as e:
        logger.error(f"❌ Image analysis error: {e}", exc_info=True)
        
        return ImageAnalysisResponse(
            success=False,
            filename="",
            error_message=str(e),
            inference_time_ms=(time.time() - start_time) * 1000
        )


# =============================================================================
# RUN SERVER
# =============================================================================

if __name__ == "__main__":
    import uvicorn
    
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
        access_log=True
    )


Overwriting main.py


In [ ]:
# =============================================================================
# FINAL STARTUP CELL - KAGGLE/JUPYTER COMPATIBLE
# =============================================================================

import os
import sys
import threading
from pyngrok import ngrok

# ============= CONFIGURE YOUR CREDENTIALS HERE =============
GROQ_API_KEY = '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'  # Your actual key
NGROK_TOKEN = '37RNvvzE8RZbzFk8N0epgQp1BbL_janF8t7LFMdtFnBJsrgc'          # Your actual token
# ===========================================================

# Set environment variables
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# Authenticate ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# Verify all files exist
required_files = ['config.py', 'groq_service.py', 'ml_model.py', 
                  'pubbert_model.py', 'llama_service.py', 'rag_service.py', 
                  'schemas.py', 'main.py']

print("Checking files...")
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"❌ Missing: {missing}. Run all %%writefile cells first!")

print("✅ All files found\n")

# Start ngrok tunnel
print("Starting ngrok tunnel...")
public_url = ngrok.connect(8000, bind_tls=True)

print("\n" + "="*70)
print("🚀 HYBRIDMED-AI v3.0")
print("="*70)
print(f"\n📡 Public URL: {public_url}")
print(f"📚 API Docs: {public_url}/docs")
print(f"💚 Health: {public_url}/health")
print("\n" + "="*70)
print("\n⏳ Loading models (2-3 minutes)...\n")

# Import app
from main import app

print("\n✅ Models loaded!")
print("\n🌐 Starting server...\n")

# Run uvicorn in a thread (Jupyter-compatible way)
import uvicorn

def run_server():
    uvicorn.run(
        app, 
        host="0.0.0.0", 
        port=8000, 
        log_level="info",
        loop="asyncio"  # Explicitly use asyncio loop
    )

# Start server in background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("\n" + "="*70)
print("✅ SERVER RUNNING")
print("="*70)
print(f"\n🌐 Access your API at: {public_url}")
print(f"📖 Documentation: {public_url}/docs")
print("\n💡 The server is running in the background.")
print("   Keep this cell running. Stop it to shut down the server.")
print("\n" + "="*70)

# Keep the cell running
try:
    server_thread.join()
except KeyboardInterrupt:
    print("\n\n🛑 Server stopped")


Checking files...
✅ All files found

Starting ngrok tunnel...

🚀 HYBRIDMED-AI v3.0

📡 Public URL: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"
📚 API Docs: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"/docs
💚 Health: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"/health


⏳ Loading models (2-3 minutes)...



2026-01-03 13:22:18,366 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
2026-01-03 13:22:19,203 - faiss.loader - INFO - Loading faiss with AVX512 support.
2026-01-03 13:22:19,230 - faiss.loader - INFO - Successfully loaded faiss with AVX512 support.
2026-01-03 13:22:19.766445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767446539.788084    1749 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767446539.794687    1749 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767446539.811486    1749 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0


✅ Models loaded!

🌐 Starting server...


✅ SERVER RUNNING

🌐 Access your API at: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"
📖 Documentation: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"/docs

💡 The server is running in the background.
   Keep this cell running. Stop it to shut down the server.



INFO:     Started server process [1749]
INFO:     Waiting for application startup.
2026-01-03 13:22:23,754 - main - INFO - ======================================================================
2026-01-03 13:22:23,754 - main - INFO - 🚀 INITIALIZING HYBRIDMED-AI v3.0
2026-01-03 13:22:23,755 - main - INFO - ======================================================================
2026-01-03 13:22:23,755 - main - INFO - 
1️⃣ Initializing Groq Service...
2026-01-03 13:22:23,829 - groq_service - INFO - ✅ Groq service initialized
2026-01-03 13:22:23,830 - main - INFO - ✅ Groq Service initialized
2026-01-03 13:22:23,830 - main - INFO - 
2️⃣ Loading ML Ensemble...
2026-01-03 13:22:23,831 - ml_model - INFO - Loading ML ensemble from: /kaggle/input/ml-ensemble/transformers/default/1/production_symptom_classifier.pkl
2026-01-03 13:22:24,558 - ml_model - INFO - ✅ ML Ensemble loaded successfully
2026-01-03 13:22:24,559 - ml_model - INFO -    • Model Type: StackingClassifier
2026-01-03 13:22:24,560 - m

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-01-03 13:22:32,244 - llama_service - INFO - ✅ Fine-tuned LLaMA loaded successfully
2026-01-03 13:22:32,245 - llama_service - INFO -    • Device: cuda
2026-01-03 13:22:32,245 - llama_service - INFO -    • Quantization: 4-bit NF4
2026-01-03 13:22:32,246 - main - INFO - 
5️⃣ Loading RAG System...
2026-01-03 13:22:32,246 - rag_service - INFO - Loading RAG system from: /kaggle/input/gale-encyclopidic-knowledge-base
2026-01-03 13:22:32,249 - rag_service - INFO - Loading FAISS index...
2026-01-03 13:22:32,309 - rag_service - INFO - Loading documents...
2026-01-03 13:22:32,517 - rag_service - INFO - Loading sentence embedder...
2026-01-03 13:22:32,520 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2026-01-03 13:22:32,521 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-01-03 13:22:33,476 - pyngrok.process.ngrok - WARNING - t=2026-01-03T13:22:33+0000 lvl=warn msg="fail

... (14 outputs left out)

INFO:     203.190.14.226:0 - "GET /health HTTP/1.1" 200 OK


2026-01-03 13:26:39,332 - pyngrok.process.ngrok - INFO - t=2026-01-03T13:26:39+0000 lvl=info msg="join connections" obj=join id=4d63e07aa6a0 l=127.0.0.1:8000 r=203.190.14.226:51330


INFO:     203.190.14.226:0 - "GET /health HTTP/1.1" 200 OK
INFO:     203.190.14.226:0 - "GET /health HTTP/1.1" 200 OK


2026-01-03 13:27:09,328 - pyngrok.process.ngrok - INFO - t=2026-01-03T13:27:09+0000 lvl=info msg="join connections" obj=join id=ce1af5c805ee l=127.0.0.1:8000 r=203.190.14.226:51330
